In [ ]:
# Clear backend issues
import os
os.environ.pop('MPLBACKEND', None)

import numpy as np
import pandas as pd
import pickle
from scipy import signal
import os
import zipfile
import urllib.request
import shutil
import cvxopt
import pyarrow
import matplotlib.pyplot as plt

# 1. Download WESAD dataset
The WESAD dataset has to be downloaded and extracted. The website is: https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download

In [13]:
# # download if not already available
# filename = 'WESAD.zip'
# if not os.path.isfile(filename):
#     print('downloading')
#     urllib.request.urlretrieve('https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download', filename)

In [14]:
# # we need to move everything inside a data folder, so all scripts work
# if not os.path.isdir('WESAD/data'):
#     os.makedirs('WESAD/data')

#     original = 'WESAD/WESAD'
#     target = 'WESAD/data/WESAD'

#     shutil.move(original, target)

# 2. Extract all data from WESAD dataset
The dataseet contains lots of data from several sources. In this project, we focus on accelerometer data from a wrist band sensor - such sensors are ubiquitous and a predition made from their data has the most business impact.

In [15]:
# to find the path to the data folder we need to get the initial directory

current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

Current directory: /Users/ashidudissanayake/Dev/Shadow/model-development


In [16]:
# get all participant folders
participants = [name for name in os.listdir('wesad/') if os.path.isdir('wesad/'+name)]
print(f"Participants found: {participants}")

Participants found: ['S5', 'S2', 'S3', 'S4', 'S17', 'S10', 'S11', 'S16', 'S8', 'S6', 'S7', 'S9', 'S13', 'S14', 'S15']


In [17]:
def get_data(data, location, sensor, columnslist):
    
    # we ge the relevant data - wrist accelerometer data and labels
    sensor_data = data['signal'][location][sensor]
    labels = data['label']
    
    # create df data and add subject column
    df = pd.DataFrame(data=sensor_data, columns=columnslist)
    df['subject'] = int(p[1:])
    
    # label is recorded in 700 HZ, resample to given frequency
    labels_resampled = signal.resample(labels, len(sensor_data))
    labels_resampled = np.rint(labels_resampled)
    labels_resampled = labels_resampled.astype(int)
    df_labels = pd.DataFrame(data=labels_resampled, columns = ['label'])
    
    #print(len_data)
    #print(len_labels)
    
    # concat df and label df
    df_res = pd.concat([df, df_labels], axis=1)

    # label definitions (see WESAD/wesad_readme.pdf)
    # 0: not defined / transient; 1: baseline; 2: stress; 3: amusement; 4: meditation; 5/6/7: should be ignored
    # we drop everything except 1 (baseline) and 2 (stress)
    df_res = df_res[(df_res['label'] == 1) | (df_res['label'] == 2)]
    
    # set the labels to 0 (no stress) and 1 (stress)
    df_res['label'] = df_res['label'].replace({1: 0, 2: 1})
        
    # remove miniscule errors introduced from resampling
    # remove rows with too few consecutive labels
    n = 10
    df_res['consec_labels'] = (df_res.groupby(['subject'])['label'].diff(1) != 0).astype('int').cumsum()
    df_res = df_res.groupby('consec_labels').filter(lambda x : len(x)>n)
    df_res = df_res.drop(columns=['consec_labels'])
    
    # split into session of specific labels
    df_res['session'] = (df_res['label'].diff() != 0).cumsum()

    return df_res

In [18]:
%%time
# look over all participant data and extract wrist acc and label data;
# combine with demographic data

cnt = 0
result_dfs_acc = []
result_dfs_bvp = []
result_dfs_eda = []
result_dfs_temp = []

for p in participants:
    
    cnt += 1
    print('Processing data: ',cnt,'/',len(participants))
    
    file = open('wesad/'+p+'/'+p+'.pkl', 'rb')
    s = pickle.load(file, encoding = 'latin1')
    
    
    df_acc = get_data(s, 'wrist', 'ACC', ['x', 'y', 'z'])
    df_bvp = get_data(s, 'wrist', 'BVP', ['BVP'])
    df_eda = get_data(s, 'wrist', 'EDA', ['EDA'])
    df_temp = get_data(s, 'wrist', 'TEMP', ['TEMP'])
    
    # assert that all features have the same number of sessions
    acc_sessions = df_acc['session'].nunique()
    assert(df_bvp['session'].nunique() == acc_sessions)
    assert(df_eda['session'].nunique() == acc_sessions)
    assert(df_temp['session'].nunique() == acc_sessions)
    
    # store results
    result_dfs_acc.append(df_acc)
    result_dfs_bvp.append(df_bvp)
    result_dfs_eda.append(df_eda)
    result_dfs_temp.append(df_temp)

Processing data:  1 / 15
Processing data:  2 / 15
Processing data:  3 / 15
Processing data:  4 / 15
Processing data:  5 / 15
Processing data:  6 / 15
Processing data:  7 / 15
Processing data:  8 / 15
Processing data:  9 / 15
Processing data:  10 / 15
Processing data:  11 / 15
Processing data:  12 / 15
Processing data:  13 / 15
Processing data:  14 / 15
Processing data:  15 / 15
CPU times: user 30 s, sys: 11 s, total: 40.9 s
Wall time: 43.2 s


In [19]:
# merge results into one df
df_acc = pd.concat(result_dfs_acc, axis=0)
df_bvp = pd.concat(result_dfs_bvp, axis=0)
df_eda = pd.concat(result_dfs_eda, axis=0)
df_temp = pd.concat(result_dfs_temp, axis=0)

In [20]:
# look into created dfs
df_acc

,x,y,z,subject,label,session
8940,63.0,4.0,9.0,5,0,1
8941,62.0,4.0,9.0,5,0,1
8942,63.0,4.0,9.0,5,0,1
8943,62.0,3.0,10.0,5,0,1
8944,62.0,4.0,10.0,5,0,1
...,...,...,...,...,...,...
121123,62.0,-13.0,1.0,15,1,2
121124,62.0,-13.0,0.0,15,1,2
121125,62.0,-13.0,0.0,15,1,2
121126,61.0,-12.0,0.0,15,1,2


In [21]:
df_bvp

,BVP,subject,label,session
17880,77.46,5,0,1
17881,78.06,5,0,1
17882,81.46,5,0,1
17883,89.68,5,0,1
17884,104.35,5,0,1
...,...,...,...,...
242250,4.95,15,1,2
242251,5.57,15,1,2
242252,6.26,15,1,2
242253,7.07,15,1,2


In [22]:
df_eda

,EDA,subject,label,session
1118,1.356115,5,0,1
1119,1.359951,5,0,1
1120,1.359951,5,0,1
1121,1.358673,5,0,1
1122,1.357394,5,0,1
...,...,...,...,...
15136,1.205489,15,1,2
15137,1.204210,15,1,2
15138,1.201651,15,1,2
15139,1.204210,15,1,2


In [23]:
df_temp

,TEMP,subject,label,session
1118,34.34,5,0,1
1119,34.34,5,0,1
1120,34.34,5,0,1
1121,34.34,5,0,1
1122,34.34,5,0,1
...,...,...,...,...
15136,29.99,15,1,2
15137,29.99,15,1,2
15138,29.99,15,1,2
15139,30.01,15,1,2


# 3. Store to Parquet file

In [24]:
# store to parquet

if not os.path.isdir('data-input'):
    os.makedirs('data-input')

df_acc.to_parquet('data-input/dataset_wesad_wrist_acc.parquet')
df_bvp.to_parquet('data-input/dataset_wesad_wrist_bvp.parquet')
df_eda.to_parquet('data-input/dataset_wesad_wrist_eda.parquet')
df_temp.to_parquet('data-input/dataset_wesad_wrist_temp.parquet')